In [1]:
from data.real.DataPreparation import DataPreparation
from evaluation.evalutation import eval
from utils import *
import torch
import os
import yaml
import pandas as pd
import random
import numpy as np
import sys
import random
from torch import nn, optim
from typing import List, Tuple
import plotly.graph_objects as go
from ucimlrepo import fetch_ucirepo

import warnings

warnings.filterwarnings("ignore")

/Users/mattia.sabella/PhD - PoliMi/Frog-DQ/Repository/FrogDQ.Federated_Proximal_Gating_with_Data_Quality/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# fetch dataset 
cdc_diabetes_health_indicators = fetch_ucirepo(id=891) 
  
# data (as pandas dataframes) 
df = cdc_diabetes_health_indicators.data.features 
df_label = cdc_diabetes_health_indicators.data.targets
df["Diabates_binary"] = df_label.iloc[:,0]
df = df.sample(frac=1).reset_index(drop=True) 

In [ ]:
display(df.head())

,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabates_binary
0,1,1,1,40,1,0,0,0,0,1,0,1,0,5,18,15,1,0,9,4,3,0
1,0,0,0,25,1,0,0,1,0,0,0,0,1,3,0,0,0,0,7,6,1,0
2,1,1,1,28,0,0,0,0,1,0,0,1,1,5,30,30,1,0,9,4,8,0
3,1,0,1,27,0,0,0,1,1,1,0,1,0,2,0,0,0,0,11,3,6,0
4,1,1,1,24,0,0,0,1,1,1,0,1,0,2,3,0,0,0,11,5,4,0


In [ ]:
display(df_label.head())

,Diabetes_binary
0,0
1,0
2,0
3,0
4,0


In [ ]:
df["Diabates_binary"] = df_label.iloc[:,0]

df.head()

,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabates_binary
0,1,1,1,40,1,0,0,0,0,1,0,1,0,5,18,15,1,0,9,4,3,0
1,0,0,0,25,1,0,0,1,0,0,0,0,1,3,0,0,0,0,7,6,1,0
2,1,1,1,28,0,0,0,0,1,0,0,1,1,5,30,30,1,0,9,4,8,0
3,1,0,1,27,0,0,0,1,1,1,0,1,0,2,0,0,0,0,11,3,6,0
4,1,1,1,24,0,0,0,1,1,1,0,1,0,2,3,0,0,0,11,5,4,0


In [6]:
print(f"Target class balance: {y.value_counts()}")

Target class balance: Diabetes_binary
0                  218334
1                   35346
Name: count, dtype: int64


In [3]:
def __undersampling(df: pd.DataFrame, label_col: str):
        df_1 = df[df[label_col] == 1]
        df_0 = df[df[label_col] == 0]

        # find the minority class size
        min_size = min(len(df_1), len(df_0))

        # sample both to the same size (undersampling majority)
        df_1_balanced = df_1.sample(n=min_size)
        df_0_balanced = df_0.sample(n=min_size)

        # combine back into a balanced dataframe
        df_balanced = pd.concat([df_1_balanced, df_0_balanced]).sample(frac=1, ignore_index=True)

        return df_balanced

df_test = df[:(int)(0.2*df.shape[0])]
df_train = df.drop(df_test.index)

#Undersampling
df_train = __undersampling(df=df_train, label_col="Diabates_binary")

print(f"Target class balance: {df_train["Diabates_binary"].value_counts()}")

Target class balance: Diabates_binary
0    28239
1    28239
Name: count, dtype: int64
